# 📊 Segment Analysis

Análisis de rendimiento del modelo por segmento (región, zona, actividad, etc.).

**Qué hace esta notebook:**
- Carga el reporte JSON de evaluación (con métricas segmentadas)
- Carga el CSV de predicciones de inferencia
- Compara métricas (AUC, precisión, recall, F1) entre segmentos
- Detecta segmentos donde el modelo rinde peor (sesgos)
- Visualiza con heatmaps, barras agrupadas y gráficos de dispersión
- Genera recomendaciones accionables por segmento

**Cuándo usarla:**
- Después de correr `energizados run train` con `segmented_evaluation` habilitado
- Antes de decidir thresholds operativos por región
- Para auditar equidad del modelo entre grupos de clientes

## 0. Configuración

In [ ]:
# ============================================================
# CONFIGURACIÓN — paths, archivos y parámetros
# ============================================================
# Toda la configuración está acá. Para apuntar la notebook a otro
# proyecto o dataset, modificá estos valores (no hace falta tocar el
# resto de las celdas).

from pathlib import Path

# --- Proyecto y datos ---
PROJECT_PATH = Path('<<PROJECT_PATH>>')
OUTPUT_PATH = PROJECT_PATH / 'output'
VERSION = '<<VERSION>>'
TRAIN_DIR = '<<TRAIN_DIR>>'
INFERENCE_DIR = '<<INFERENCE_DIR>>'

# --- Reporte de evaluación (run de entrenamiento, con métricas segmentadas) ---
EVALUATION_REPORT = OUTPUT_PATH / VERSION / TRAIN_DIR / 'reports' / 'evaluation' / 'evaluation_report.json'

# --- Predicciones de inferencia (para análisis de distribución por segmento) ---
PREDICTIONS_CSV = OUTPUT_PATH / VERSION / INFERENCE_DIR / 'predictions.csv'

# --- Entorno (detección automática Colab vs local) ---
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# Segmentos a analizar (se toman del reporte automáticamente si están disponibles)
# Dejá la lista vacía para usar todos los segmentos del reporte
SEGMENTS_TO_ANALYZE = []  # ej: ["geo_region"] — vacío = todos los disponibles

print(f'PROJECT_PATH      : {PROJECT_PATH}')
print(f'EVALUATION_REPORT : {EVALUATION_REPORT}')
print(f'PREDICTIONS_CSV   : {PREDICTIONS_CSV}')
print(f'IN_COLAB          : {IN_COLAB}')

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120
plt.rcParams["figure.figsize"] = (12, 6)

# Verificar que el reporte existe
assert Path(EVALUATION_REPORT).exists(), f"No se encuentra: {EVALUATION_REPORT}"
print("✅ Configuración lista")

## 1. Carga y exploración del reporte

El reporte de evaluación (JSON) contiene métricas globales y, si `segmented_evaluation` estaba habilitado en `train.yaml`, métricas por segmento.

**Qué mirar:**
- `segmented_metrics`: claves como `geo_region`, `geo_cluster`, `zona` — cada una es una columna por la que se segmentó.
- Cada segmento dentro tiene: `n_samples` (cuántos clientes), `n_positives` (cuántos fraudes reales), `positive_rate` (tasa de fraude), `auc`, `precision`, `recall`, `f1`, y `threshold`.

In [ ]:
with open(EVALUATION_REPORT) as f:
    report = json.load(f)

print("=== Top-level keys ===")
for k in report.keys():
    print(f"  {k}")

print(f"\nTimestamp: {report.get('timestamp', 'N/A')}")

# Métricas globales
global_metrics = report.get("metrics", {})
print(f"\n=== Métricas globales ===")
for k, v in global_metrics.items():
    if not isinstance(v, (dict, list)):
        print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

# Segmentos disponibles
segmented = report.get("segmented_metrics", {})
print(f"\n Segmentos disponibles: {list(segmented.keys())}")
for seg_col, segments in segmented.items():
    print(f"  {seg_col}: {len(segments)} valores → {list(segments.keys())[:5]}...")

## 2. Armado de la tabla de segmentos

Aplanamos las métricas segmentadas en un DataFrame para análisis y visualización.

**Estructura de la tabla:**
- `segment_column`: qué columna se usó para segmentar (ej. `geo_region`)
- `segment_value`: valor del segmento (ej. `FLORIANOPOLIS`)
- Métricas: `n_samples`, `positive_rate`, `auc`, `precision`, `recall`, `f1`, `threshold`

**Qué mirar en esta tabla:**
- Segmentos con `n_samples` muy chicos → métricas poco confiables (pocos datos).
- `positive_rate` muy variable entre segmentos → el modelo puede tener sesgo.
- `auc` bajo (< 0.6) → el modelo no discrimina bien en ese segmento.
- `f1` bajo → el balance precisión-recall es pobre.

In [ ]:
# Construir tabla plana de segmentos
rows = []
segments_to_use = SEGMENTS_TO_ANALYZE if SEGMENTS_TO_ANALYZE else list(segmented.keys())

for seg_col in segments_to_use:
    if seg_col not in segmented:
        print(f"⚠️  '{seg_col}' no está en el reporte. Disponibles: {list(segmented.keys())}")
        continue
    for seg_value, metrics in segmented[seg_col].items():
        rows.append({
            "segment_column": seg_col,
            "segment_value": seg_value,
            **{k: v for k, v in metrics.items() if k != "threshold_mode"},
        })

seg_df = pd.DataFrame(rows)
print(f"Total de filas: {len(seg_df)}")
print(f"Columnas de segmento: {seg_df['segment_column'].unique()}")
seg_df.head(10)

## 3. Heatmap de métricas por segmento

Un heatmap permite ver de un golpe de vista qué segmentos están peor en cada métrica.

**Qué mirar:**
- **Columnas rojas en AUC:** segmentos donde el modelo no separa bien fraude de no-fraude. Si un segmento tiene AUC < 0.55, el modelo es apenas mejor que aleatorio.
- **Columnas rojas en F1:** segmentos con bajo rendimiento combinado. Puede ser por pocos datos de entrenamiento, patrones distintos, o features que no generalizan.
- **Segmentos con `positive_rate` muy distinto al promedio:** posible Sesgo de selección en los datos de entrenamiento.
- **Filas enteras rojas:** segmentos problemáticos que necesitan investigación dedicada (más datos, features específicas, o modelo separado).

In [ ]:
# Heatmap de métricas por segmento
for seg_col in seg_df["segment_column"].unique():
    sub = seg_df[seg_df["segment_column"] == seg_col].set_index("segment_value")
    metric_cols = ["auc", "precision", "recall", "f1", "positive_rate"]
    heatmap_data = sub[[c for c in metric_cols if c in sub.columns]]
    
    if heatmap_data.empty:
        continue
    
    fig, ax = plt.subplots(figsize=(max(8, len(heatmap_data.columns) * 2.5),
                                    max(5, len(heatmap_data) * 0.5)))
    sns.heatmap(heatmap_data, annot=True, fmt=".3f", cmap="RdYlGn",
                center=0.5, vmin=0, vmax=1, ax=ax,
                cbar_kws={"label": "Valor"})
    ax.set_title(f"Métricas por {seg_col}", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()

## 4. Barras comparativas: AUC y F1 por segmento

Los gráficos de barras ordenados de peor a mejor muestran claramente la disparidad entre segmentos.

**Qué mirar:**
- **Diferencia entre el mejor y peor segmento en AUC:** si es > 0.2, hay un problema de equidad serio. El modelo funciona bien para unos clientes y mal para otros.
- **Barras muy cortas (AUC < 0.6):** priorizá estos segmentos para mejora.
- **Barras sin datos (n_samples < 30):** segmentos con muy pocos ejemplos. Las métricas no son confiables — necesitás más datos antes de concluir.
- **Línea punteada:** representa la métrica global. Idealmente las barras deberían estar cerca de esa línea (equidad horizontal).

In [ ]:
# Barras agrupadas: AUC y F1 por segmento, ordenadas de peor a mejor
for seg_col in seg_df["segment_column"].unique():
    sub = seg_df[seg_df["segment_column"] == seg_col].sort_values("auc")
    
    fig, axes = plt.subplots(1, 2, figsize=(16, max(5, len(sub) * 0.4)))
    
    # AUC
    colors_auc = ["#F44336" if v < 0.6 else "#FF9800" if v < 0.7 else "#4CAF50" for v in sub["auc"]]
    axes[0].barh(sub["segment_value"], sub["auc"], color=colors_auc, edgecolor="white")
    axes[0].axvline(global_metrics.get("auc", 0.5), color="gray", linestyle="--", linewidth=1.5, label=f"Global AUC={global_metrics.get('auc', 0):.3f}")
    axes[0].set_title(f"AUC por {seg_col}")
    axes[0].set_xlabel("AUC")
    axes[0].set_xlim(0, 1)
    axes[0].legend(loc="lower right", fontsize=8)
    
    # F1
    colors_f1 = ["#F44336" if v < 0.3 else "#FF9800" if v < 0.5 else "#4CAF50" for v in sub["f1"]]
    axes[1].barh(sub["segment_value"], sub["f1"], color=colors_f1, edgecolor="white")
    axes[1].axvline(global_metrics.get("f1", 0.5), color="gray", linestyle="--", linewidth=1.5, label=f"Global F1={global_metrics.get('f1', 0):.3f}")
    axes[1].set_title(f"F1 por {seg_col}")
    axes[1].set_xlabel("F1")
    axes[1].set_xlim(0, 1)
    axes[1].legend(loc="lower right", fontsize=8)
    
    plt.tight_layout()
    plt.show()

## 5. Tamaño del segmento vs rendimiento

¿Los segmentos con más datos rinden mejor? Este gráfico responde esa pregunta.

**Qué mirar:**
- **Tendencia creciente (más samples = mejor AUC):** el modelo necesita más datos para generalizar. Segmentos con pocos datos están perjudicados.
- **Outliers arriba a la izquierda (pocos datos, buen AUC):** posible sobreajuste — chequear que la métrica no esté inflada por pocos ejemplos.
- **Outliers abajo a la derecha (muchos datos, mal AUC):** segmento problemático — el modelo no logra aprender sus patrones incluso con datos suficientes. Posiblemente necesite features específicas.
- **Tamaño de la burbuja = positive_rate:** burbujas grandes = segmentos con alta tasa de fraude.

In [ ]:
# Scatter: n_samples vs AUC, coloreado por segment_column, tamaño = positive_rate
fig, ax = plt.subplots(figsize=(12, 7))

for seg_col in seg_df["segment_column"].unique():
    sub = seg_df[seg_df["segment_column"] == seg_col]
    sizes = (sub["positive_rate"] * 500).clip(lower=20)
    ax.scatter(
        sub["n_samples"], sub["auc"],
        s=sizes, alpha=0.7, edgecolors="white", linewidth=0.5,
        label=seg_col
    )
    # Etiquetar segmentos extremos
    for _, row in sub.iterrows():
        if row["auc"] < 0.6 or row["auc"] > 0.85:
            ax.annotate(row["segment_value"], (row["n_samples"], row["auc"]),
                       fontsize=7, alpha=0.8,
                       xytext=(5, 5), textcoords="offset points")

ax.axhline(global_metrics.get("auc", 0.5), color="gray", linestyle="--", alpha=0.5)
ax.set_xlabel("Número de samples en el segmento")
ax.set_ylabel("AUC")
ax.set_title("Tamaño del segmento vs AUC (burbuja = tasa de fraude)")
ax.set_xlim(left=0)
ax.set_ylim(0, 1)
ax.legend(loc="lower right", fontsize=8)
plt.tight_layout()
plt.show()

## 6. Precision vs Recall por segmento

Cada punto es un segmento. La posición muestra el trade-off precisión-recall. El tamaño es la cantidad de samples.

**Qué mirar:**
- **Esquina superior derecha (alta precisión, alto recall):** segmentos ideales — el modelo acierta y no se pierde fraudes.
- **Esquina inferior derecha (alta precisión, bajo recall):** el modelo es conservador — solo marca cuando está muy seguro, pero deja pasar muchos fraudes.
- **Esquina superior izquierda (baja precisión, alto recall):** el modelo es agresivo — marca muchos, pero con muchos falsos positivos.
- **Esquina inferior izquierda:** mal en ambas — el segmento más problemático.
- **Línea diagonal punteada:** F1 constante. Los puntos sobre la misma curva tienen el mismo F1.

In [ ]:
# Precision vs Recall con curvas de iso-F1
fig, ax = plt.subplots(figsize=(10, 8))

# Curvas iso-F1
for f1_level in [0.1, 0.2, 0.3, 0.5, 0.7, 0.9]:
    p = np.linspace(0.01, 1, 100)
    # Evitar división por cero cuando 2*p == f1_level
    with np.errstate(divide='ignore', invalid='ignore'):
        r = np.where(np.isclose(2 * p, f1_level), np.nan, (f1_level * p) / (2 * p - f1_level))
    valid = (r > 0) & (r <= 1)
    ax.plot(p[valid], r[valid], "gray", alpha=0.2, linewidth=0.5)
    # Label at rightmost point
    if valid.any():
        idx = valid.nonzero()[0][-1]
        ax.annotate(f"F1={f1_level}", (p[idx], r[idx]), fontsize=7, alpha=0.4)

for seg_col in seg_df["segment_column"].unique():
    sub = seg_df[seg_df["segment_column"] == seg_col]
    ax.scatter(sub["precision"], sub["recall"],
              s=sub["n_samples"] / sub["n_samples"].max() * 300 + 30,
              alpha=0.7, edgecolors="white", linewidth=0.5, label=seg_col)
    for _, row in sub.iterrows():
        ax.annotate(row["segment_value"], (row["precision"], row["recall"]),
                   fontsize=7, alpha=0.8, xytext=(5, 5), textcoords="offset points")

ax.set_xlabel("Precision")
ax.set_ylabel("Recall")
ax.set_title("Precision vs Recall por segmento (tamaño = n_samples)")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 7. Distribución de probabilidades por segmento (requiere predictions CSV)

Si cargaste el CSV de predicciones, esta sección muestra cómo se distribuyen las probabilidades dentro de cada segmento.

**Qué mirar:**
- **Boxplots desplazados hacia arriba:** el modelo asigna probabilidades sistemáticamente más altas a ese segmento → posible sesgo.
- **Boxplots muy anchos:** el modelo tiene opiniones muy variadas dentro del segmento → posiblemente hay subgrupos no capturados.
- **Boxplots muy angostos:** el modelo colapsa todas las predicciones a un valor similar → no está discriminando dentro del segmento.

In [ ]:
predictions_path = Path(PREDICTIONS_CSV)
if predictions_path.exists():
    pred_df = pd.read_csv(predictions_path)
    print(f"Predicciones cargadas: {len(pred_df):,} filas")
    
    # Detectar columnas de segmento presentes en las predicciones
    seg_cols_in_preds = [c for c in seg_df["segment_column"].unique() if c in pred_df.columns]
    
    if seg_cols_in_preds:
        for seg_col in seg_cols_in_preds[:2]:  # máximo 2 para no saturar
            fig, ax = plt.subplots(figsize=(max(8, pred_df[seg_col].nunique() * 0.8), 5))
            order = pred_df.groupby(seg_col)["probability"].median().sort_values().index
            sns.boxplot(data=pred_df, x=seg_col, y="probability", order=order,
                       hue=seg_col, palette="RdYlGn", legend=False, ax=ax)
            ax.axhline(0.5, color="crimson", linestyle="--", linewidth=1, label="threshold=0.5")
            ax.set_title(f"Distribución de probabilidad por {seg_col}")
            ax.set_xlabel(seg_col)
            ax.set_ylabel("Probability")
            plt.xticks(rotation=45, ha="right")
            ax.legend()
            plt.tight_layout()
            plt.show()
    else:
        print("⚠️  Las columnas de segmento del reporte no están en el CSV de predicciones.")
        print(f"   Columnas en predicciones: {list(pred_df.columns)}")
else:
    print("ℹ️  No se encontró el CSV de predicciones. Saltando análisis de distribución.")
    print(f"   Path buscado: {predictions_path.resolve()}")

## 8. Matriz de equidad: diferencias vs el global

Calculamos cuánto se desvía cada segmento de la métrica global. Valores negativos = peor que el promedio.

**Qué mirar:**
- **Δ AUC negativo grande:** segmentos donde el modelo rinde mucho peor que el promedio. Prioridad de mejora.
- **Δ F1 negativo grande:** posiblemente por bajo recall (no encuentra fraudes) o baja precisión (muchos falsos positivos).
- **Patrón sistemático:** si ciertos valores de segmento consistentemente tienen Δ negativo en todas las métricas, es un sesgo estructural del modelo.

In [ ]:
# Calcular diferencias vs métrica global
global_auc = global_metrics.get("auc", 0.5)
global_f1 = global_metrics.get("f1", 0.5)
global_precision = global_metrics.get("precision", 0.5)
global_recall = global_metrics.get("recall", 0.5)

equity_rows = []
for seg_col in seg_df["segment_column"].unique():
    sub = seg_df[seg_df["segment_column"] == seg_col]
    for _, row in sub.iterrows():
        equity_rows.append({
            "segment_column": seg_col,
            "segment_value": row["segment_value"],
            "Δ AUC": row["auc"] - global_auc,
            "Δ F1": row["f1"] - global_f1,
            "Δ Precision": row["precision"] - global_precision,
            "Δ Recall": row["recall"] - global_recall,
            "n_samples": row["n_samples"],
        })

equity_df = pd.DataFrame(equity_rows).sort_values("Δ AUC")

fig, ax = plt.subplots(figsize=(12, max(5, len(equity_df) * 0.35)))

x = np.arange(len(equity_df))
width = 0.2
for i, (col, color) in enumerate(zip(
    ["Δ AUC", "Δ F1", "Δ Precision", "Δ Recall"],
    ["#2196F3", "#4CAF50", "#FF9800", "#9C27B0"]
)):
    ax.barh(x + i * width, equity_df[col], width, label=col, color=color, alpha=0.8)

ax.set_yticks(x + width * 1.5)
ax.set_yticklabels(equity_df["segment_value"])
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Diferencia vs métrica global")
ax.set_title("Equidad: desviación de cada segmento vs el promedio global")
ax.legend(loc="lower right", fontsize=8)
plt.tight_layout()
plt.show()

# Tabla de los peores segmentos
print("=== Segmentos más por debajo del promedio (bottom 5 por Δ AUC) ===")
display(equity_df.head(5))

## 8b. Matriz de confusión por segmento

La matriz de equidad (sección 8) muestra **cuánto** se desvía cada segmento del global. Esta sección va un nivel más profundo: **¿dónde se equivoca el modelo en cada segmento?**

**Interpretación:**
- **Falsos positivos (FP):** clientes legítimos marcados como fraude. Cada FP es una inspección desperdiciada. Segmentos con alto FP → el modelo está "acusando" de más.
- **Falsos negativos (FN):** fraudes que el modelo no detectó. Segmentos con alto FN → hay fraude que se está escapando. Prioridad de mejora.
- **Specificity (TN / (TN + FP)):** qué tan bien el modelo identifica a los legítimos. Baja specificity = muchas falsas alarmas.
- **NPV (TN / (TN + FN)):** si el modelo dice "no es fraude", ¿qué tan probable es que acierte?

> Derivamos TP, FP, FN, TN a partir de `precision`, `recall`, `n_positives` y `n_samples`, que son las métricas que el reporte de evaluación ya calcula por segmento.

In [ ]:
# --- Matriz de confusión derivada por segmento ---
# A partir de precision, recall, n_positives y n_samples, derivamos TP, FP, FN, TN

def derive_confusion_matrix(row):
    """Deriva la matriz de confusión de las métricas agregadas por segmento."""
    n = row['n_samples']
    p = row['n_positives']  # positivos reales
    precision = row['precision']
    recall = row['recall']

    # Evitar división por cero
    if p == 0 or precision == 0 or recall == 0:
        return pd.Series({'TP': 0, 'FP': 0, 'FN': p, 'TN': n - p,
                         'specificity': np.nan, 'FPR': np.nan, 'NPV': np.nan})

    TP = recall * p
    FP = (TP / precision) - TP if precision > 0 else 0
    FN = p - TP
    TN = n - TP - FP - FN

    specificity = TN / (TN + FP) if (TN + FP) > 0 else np.nan
    FPR = FP / (FP + TN) if (FP + TN) > 0 else np.nan
    NPV = TN / (TN + FN) if (TN + FN) > 0 else np.nan

    return pd.Series({
        'TP': int(round(TP)), 'FP': int(round(FP)),
        'FN': int(round(FN)), 'TN': int(round(TN)),
        'specificity': specificity, 'FPR': FPR, 'NPV': NPV
    })

# Aplicar a todos los segmentos
cm_cols = seg_df.apply(derive_confusion_matrix, axis=1)
seg_cm = pd.concat([seg_df, cm_cols], axis=1)

# Tabla resumen: métricas de error por segmento
error_summary = seg_cm[['segment_column', 'segment_value', 'n_samples', 'n_positives',
                         'precision', 'recall', 'f1', 'specificity', 'FPR', 'NPV',
                         'TP', 'FP', 'FN', 'TN']].copy()

print('=== Métricas de error por segmento (ordenado por FPR descendente) ===')
try:
    display(error_summary.sort_values('FPR', ascending=False)
    .style.format({
        'precision': '{:.3f}', 'recall': '{:.3f}', 'f1': '{:.3f}',
        'specificity': '{:.3f}', 'FPR': '{:.3f}', 'NPV': '{:.3f}'
    })
    .background_gradient(cmap='RdYlGn_r', subset=['FPR'])
    .background_gradient(cmap='RdYlGn', subset=['specificity', 'NPV'])  # noqa: E501
)  # noqa: E501

except (AttributeError, ImportError):
    # Fallback: si jinja2 no está instalado, mostrar tabla simple
    print(error_summary.sort_values('FPR', ascending=False).to_string(index=False))

# Visualización: heatmap de FP rate y FN rate por segmento
fig, axes = plt.subplots(1, 2, figsize=(16, max(5, len(seg_cm) * 0.4)))

# Preparar datos para heatmap
heatmap_data = seg_cm.set_index('segment_value')[['FPR', 'recall']].copy()
heatmap_data.columns = ['Tasa Falsos Positivos', 'Recall (TPR)']

sns.heatmap(heatmap_data, annot=True, fmt='.3f', cmap='RdYlGn_r',
            cbar_kws={'label': 'Tasa'}, ax=axes[0], vmin=0, vmax=1,
            linewidths=0.5)
axes[0].set_title('FPR y Recall por segmento')

# Scatter: FPR vs Recall — el trade-off real
for seg_col in seg_cm['segment_column'].unique():
    sub = seg_cm[seg_cm['segment_column'] == seg_col]
    axes[1].scatter(sub['FPR'], sub['recall'], s=sub['n_samples'] / 50, alpha=0.7,
                   label=seg_col)
    for _, row in sub.iterrows():
        axes[1].annotate(row['segment_value'][:12],
                        (row['FPR'], row['recall']), fontsize=7, alpha=0.8)

axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('Recall (True Positive Rate)')
axes[1].set_title('Trade-off FPR vs Recall por segmento')
axes[1].legend(fontsize=8, loc='lower right')
axes[1].set_xlim(0, 1)
axes[1].set_ylim(0, 1)
axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.3)
plt.tight_layout()
plt.show()

# Resumen narrativo
print(f'\n--- Resumen de errores por segmento ---')
worst_fpr = error_summary.sort_values('FPR', ascending=False).iloc[0]
worst_fnr = error_summary.sort_values('recall').iloc[0]  # menor recall = mayor FN rate
print(f'Mayor tasa de falsos positivos: {worst_fpr["segment_value"]} '
      f'(FPR={worst_fpr["FPR"]:.3f}, {worst_fpr["FP"]} de {worst_fpr["n_samples"]} clientes)')
print(f'Mayor tasa de falsos negativos:  {worst_fnr["segment_value"]} '
      f'(Recall={worst_fnr["recall"]:.3f}, {worst_fnr["FN"]} fraudes no detectados de {worst_fnr["n_positives"]})')


## 9. Recomendaciones

Basado en el análisis, generamos recomendaciones automáticas por segmento.

In [ ]:
print("=== RECOMENDACIONES POR SEGMENTO ===\n")

for _, row in seg_df.iterrows():
    issues = []
    if row["auc"] < 0.55:
        issues.append("⚠️  AUC crítico (< 0.55): modelo apenas mejor que aleatorio")
    elif row["auc"] < 0.65:
        issues.append("🔍 AUC bajo (< 0.65): considerar features específicas para este segmento")
    
    if row["n_samples"] < 100:
        issues.append(f"📉 Pocos samples ({row['n_samples']}): métricas poco confiables, recolectar más datos")
    
    if row["recall"] < 0.5:
        issues.append("🔴 Recall bajo (< 0.5): más de la mitad de los fraudes no se detectan")
    
    if row["precision"] < 0.3:
        issues.append("🟡 Precisión baja (< 0.3): muchos falsos positivos, costo operativo alto")
    
    if row["f1"] < 0.3:
        issues.append("🔴 F1 crítico (< 0.3): rendimiento global muy pobre")
    
    if row["positive_rate"] < 0.01:
        issues.append("⚪ Tasa de fraude muy baja (< 1%): el modelo puede ignorar este segmento")
    
    if issues:
        print(f"[{row['segment_column']}={row['segment_value']}] n={row['n_samples']}, AUC={row['auc']:.3f}, F1={row['f1']:.3f}")
        for issue in issues:
            print(f"  {issue}")
        print()

# Resumen global
n_problematic = len(seg_df[seg_df["auc"] < 0.6])
n_total = len(seg_df)
print(f"\nResumen: {n_problematic}/{n_total} segmentos con AUC < 0.6")
print(f"Segmentos críticos (AUC < 0.55): {len(seg_df[seg_df['auc'] < 0.55])}")
print(f"Segmentos con pocos datos (n < 100): {len(seg_df[seg_df['n_samples'] < 100])}")

## 10. Export

Exporta las tablas de segmentos a CSV para incluir en informes.

In [ ]:
export_dir = Path(EVALUATION_REPORT).parent / "segment_analysis"
export_dir.mkdir(exist_ok=True)

seg_df.to_csv(export_dir / "segment_metrics.csv", index=False)
print(f"✅ {export_dir / 'segment_metrics.csv'}")

equity_df.to_csv(export_dir / "equity_gaps.csv", index=False)
print(f"✅ {export_dir / 'equity_gaps.csv'}")

print(f"\nArchivos exportados a: {export_dir.resolve()}")

---
## Notas

- El reporte de evaluación con segmentos requiere que `train.yaml` tenga `segmented_evaluation` configurado. Ejemplo:
  ```yaml
  evaluation:
    segmented_evaluation:
      segment_columns: ["geo_region", "zona"]
  ```
- Si no tenés segmentos en el reporte, esta notebook solo mostrará las métricas globales.
- Las recomendaciones son heurísticas — siempre validalas con conocimiento de dominio.